<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/groundwatfil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Infiltration - Trickling Filtration calculations***

In [ ]:
import math as mh
import pandas as pd

def trickling(lanes,Partable2,dailyflow,Treatment):
    print("=== Trickling Filter Design for Ammonia Removal ===")

    # User Inputs
    Cin = float(input("Enter influent ammonia concentration (mg/L as NH₄⁺-N): "))
    Cout = float(input("Enter target effluent ammonia concentration (mg/L as NH₄⁺-N default: 1 mg/L): ") or 1)
    HLR = float(input("Enter hydraulic loading rate (m³/m²·h, default=1.5): ") or 1.5)
    O2_req = float(input("Enter oxygen requirement (kg O₂/kg NH₄⁺-N, default=4.6 maximum 5.5): ") or 4.6)
    energy_per_O2 = float(input("Enter energy consumption (kWh/kg O₂, default=1.5): ") or 1.5)
    flowlane=dailyflow/lanes # flow per lane in m3/day


    #  Ammonia and Oxygen demand Calculations
    ammonia_removed = flowlane * (Cin - Cout) / 1000  # kg/h
    O2_demand = ammonia_removed * O2_req  # kg O₂/h

    #energy_hourly = O2_demand * energy_per_O2  # kWh/h
    #energy_daily = energy_hourly * 24  # kWh/da

    # Trickling filter area
    area = Q / HLR  # m²
    area = mh.ceil(area)
    diameter = (area * 4 / 3.1416) ** 0.5  # Assumes circular filter
    diameter = mh.ceil(diameter)
    trickarea=lanes*area #m2

    # Blower specifications
    print("The mass of O₂ in air ≈ 0.23 kg O₂/kg air")
    oxygen_mass_fraction_in_air = 0.23  # 23% mass O2
    print("Air density at normal conditions is assumed ≈ 1.293 kg/Nm³.")
    air_density_Nm3 = 1.293  # kg/m³ at normal conditions
    # Step 3: Air needed without considering transfer efficiency
    air_needed_kgd = O2_demand / oxygen_mass_fraction_in_air  # kg air/day
    # Convert to Nm³/day
    air_needed_Nm3d = air_needed_kgd / air_density_Nm3
    # Correct for oxygen transfer efficiency
    ote = float(input("Set the oxygen transfer efficiency (OTE) (0-1): "))
    real_air_needed_Nm3d = air_needed_Nm3d / ote
    # Airflow per hour
    airflow_Nm3h = real_air_needed_Nm3d / 24  # Nm³/h
    Q_m3s = airflow_Nm3h / 3600  # m³/s
    pressure_loss_pa = float(input("Static pressure loss (across media and piping) ≈ 2000–4000 Pa (default: 3000): ") or 3000)
    efficiency=0.7
    # Blower power requirement
    blower_power_W = (pressure_loss_pa * Q_m3s) / efficiency  # Watts
    blower_power_W=mh.ceil(blower_power_W)
    # daily energy consumption per lane
    trickenergylane=(blower_power_W * 24) / 1000  # kWh/day
    trickenergylane=mh.ceil(trickenergylane)
    # Daily energy consumption
    trickenergy=trickenergylane*lanes
    trickenergy=mh.ceil(trickenergy)

    # Results
    print("\n=== Results ===")
    print(f"Ammonia removed: {ammonia_removed:.3f} kg/h")
    print(f"Oxygen demand: {O2_demand:.3f} kg O₂/h")
    print(f"Total area of Trickling filters: {trickarea:.2f} m²")
    print(f"Approximate diameter per trickling filters (circular): {diameter:.2f} m")
    print(f"Required Airflow: {airflow_Nm3h:.2f} Nm³/h")
    print(f"Required Blower Power (1+1 stand-by per lane): {blower_power_W:.2f} W")
    print(f"Daily Energy Consumption per lane: {trickenergylane:.2f} kWh/day")
    print(f"Daily Energy Consumption of the trickling filtration: {trickenergy:.2f} kWh/day")

    return trickarea,trickenergy